## 1. Mathematical Preliminaries & Graph Theory

### 1.1 Microgrid as a Graph

A DC microgrid is formally represented as an undirected graph:

$$G = (V, E, W)$$

where:
- $V = \{v_1, v_2, \ldots, v_n\}$: Set of $n$ nodes (buses/vertices)
- $E \subseteq V \times V$: Set of $m$ edges (transmission lines/connections)
- $W: E \rightarrow \mathbb{R}^+$: Weight function assigning impedance to edges

For each edge $e = (i,j) \in E$, the impedance is:

$$Z_{ij} = R_{ij} + jX_{ij}$$

where:
- $R_{ij}$: Series resistance (ohms) - determines resistive losses
- $X_{ij}$: Series reactance (ohms) - determines reactive power behavior

### 1.2 Adjacency & Admittance Matrices

The **adjacency matrix** $A$ captures connectivity:

$$A_{ij} = \begin{cases}
1 & \text{if } (i,j) \in E \\
0 & \text{otherwise}
\end{cases}$$

The **nodal admittance matrix** (bus admittance matrix) is:

$$Y_{bus,ii} = \sum_{j \neq i} Y_{ij} \quad \text{(diagonal - self admittance)}$$

$$Y_{bus,ij} = -Y_{ij} \quad \text{(off-diagonal - mutual admittance)}$$

where admittance is:

$$Y_{ij} = \frac{1}{Z_{ij}} = \frac{1}{R_{ij} + jX_{ij}}$$

### 1.3 Topology Metrics

**Degree Distribution**:
$$d_i = |\{j : (i,j) \in E\}|$$

**Network Density**:
$$\rho = \frac{2|E|}{|V|(|V|-1)}$$

**Clustering Coefficient** (local triangulation):
$$C_i = \frac{2t_i}{d_i(d_i-1)}$$
where $t_i$ is the number of triangles containing node $i$.

**Average Path Length**:
$$L = \frac{1}{n(n-1)} \sum_{i \neq j} d_{ij}$$
where $d_{ij}$ is the shortest path distance between nodes $i$ and $j$.

## 2. Canonical Topology Architectures

### 2.1 Mesh Topology (Complete Graph)

**Definition**: A complete graph $K_n$ where every pair of nodes is directly connected.

**Properties**:
- **Edges**: $m = \frac{n(n-1)}{2}$
- **Degree (uniform)**: $d_i = n-1$ for all $i$
- **Density**: $\rho = 1$ (maximum connectivity)
- **Diameter**: 1 (every node is one hop away)
- **Clustering coefficient**: $C = 1$ (complete transitivity)

**Electrical Characteristics**:
- **Redundancy**: $(n-1)$ parallel paths between any two nodes
- **Voltage regulation**: Superior - multiple paths distribute current
- **Loss sensitivity**: Lower - multiple parallel routes reduce line congestion
- **Fault tolerance**: Excellent - can tolerate up to $n-2$ line failures while maintaining connectivity

**Constraints**:
- **Cost**: $O(n^2)$ - prohibitively expensive for large networks
- **Complexity**: $O(n^2)$ parameters to specify
- **Scalability**: Limited to small networks (typically $n \leq 20$)

### 2.2 Ring Topology (Cycle Graph)

**Definition**: A cycle graph $C_n$ where nodes are arranged in a ring: $0 - 1 - 2 - \ldots - (n-1) - 0$.

**Properties**:
- **Edges**: $m = n$
- **Degree (uniform)**: $d_i = 2$ for all $i$
- **Density**: $\rho = \frac{2}{n-1}$ (sparse)
- **Diameter**: $\lfloor n/2 \rfloor$
- **Clustering coefficient**: $C = 0$ (no triangles)

**Electrical Characteristics**:
- **Redundancy**: 2 paths between any two nodes (ring provides redundancy)
- **Voltage regulation**: Moderate - circular flow allows some load balancing
- **Loss profile**: Concentrated on diameter nodes (bottleneck effect)
- **Fault tolerance**: Can survive single line failure; diameter path is critical

**Advantages**:
- Efficient backbone design for medium networks
- Natural feeder architecture for distribution systems
- $O(n)$ cost - scalable

### 2.3 Tree Topology (Minimum Spanning Tree)

**Definition**: An acyclic connected graph with $m = n-1$ edges. Represents hierarchical distribution.

**Properties**:
- **Edges**: $m = n-1$ (minimum for connectivity)
- **Cycles**: 0 (acyclic)
- **Density**: $\rho = \frac{2(n-1)}{n(n-1)} = \frac{2}{n}$ (highly sparse)
- **Diameter**: Varies with structure (can be $O(n)$ for linear, $O(\log n)$ for balanced)
- **Unique path**: Exactly one path exists between any two nodes

**Electrical Characteristics**:
- **Redundancy**: None - single point of failure (line disconnects the network)
- **Voltage regulation**: Poor - limited alternative paths
- **Loss concentration**: Severe - root-to-leaf paths accumulate losses
- **Fault tolerance**: Critical failures at root nodes

**Advantages**:
- **Cost**: Minimal - $O(n)$ edges
- **Scalability**: Excellent - supports thousands of nodes
- **Hierarchical structure**: Natural for distribution networks with clear source-load relationship

In [1]:
# Import required libraries
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple, Dict, Any, List
import warnings
warnings.filterwarnings('ignore')

# Configure visualization
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 10

# Add simulation module to path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from simulation.create_network import create_microgrid
from simulation.run_powerflow import run_powerflow_analysis, check_constraints

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


## 3. Implementation: Network Construction

### 3.1 Mesh Topology Implementation

We construct a complete graph $K_n$ where every node connects to every other node.

**Algorithm**:
```
Input: n (number of nodes)
1. Create complete graph: G = K_n
2. For each edge (i,j):
   - Set R_ij = 0.05 Ω (series resistance)
   - Set X_ij = 0.10 Ω (series reactance)
3. For each node i:
   - Initialize V_i = 1.0 pu (voltage magnitude)
   - Set generation/load randomly
Output: G with electrical parameters
```

**Computational Complexity**:
- Time: $O(n^2)$ for edge creation
- Space: $O(n^2)$ for adjacency matrix storage

In [ ]:
# 3.1 MESH TOPOLOGY CONSTRUCTION
# Create a complete graph K_8 with realistic electrical parameters

n_nodes_mesh = 8
mesh_graph, mesh_metadata = create_microgrid(
    num_nodes=n_nodes_mesh,
    topology='mesh'
)

print(f"Mesh Topology (K_{n_nodes_mesh}) Network Properties:")
print(f"{'─' * 50}")
print(f"Nodes (|V|):                    {mesh_graph.number_of_nodes()}")
print(f"Edges (|E|):                    {mesh_graph.number_of_edges()}")
print(f"Expected edges for K_n:         {n_nodes_mesh * (n_nodes_mesh - 1) // 2}")
print(f"Density (ρ):                    {nx.density(mesh_graph):.4f}")
print(f"Average degree:                 {2 * mesh_graph.number_of_edges() / mesh_graph.number_of_nodes():.2f}")
print(f"Diameter:                       {nx.diameter(mesh_graph)}")
print(f"Average clustering coefficient: {nx.average_clustering(mesh_graph):.4f}")
print(f"Is connected:                   {nx.is_connected(mesh_graph)}")

# Compute degree distribution
degrees = [mesh_graph.degree(n) for n in mesh_graph.nodes()]
print(f"\nDegree distribution: min={min(degrees)}, max={max(degrees)}, mean={np.mean(degrees):.2f}")

### 3.2 Ring Topology Implementation

We construct a cycle graph $C_n$ with circular arrangement.

**Algorithm**:
```
Input: n (number of nodes)
1. Create cycle: edges = {(0,1), (1,2), ..., (n-2,n-1), (n-1,0)}
2. Add optional internal chords for enhanced connectivity
3. Assign electrical parameters to edges
Output: Cycle with $n$ edges, diameter ≈ n/2
```

In [ ]:
# 3.2 RING TOPOLOGY CONSTRUCTION
# Create a cycle graph C_n

n_nodes_ring = 12
ring_graph, ring_metadata = create_microgrid(
    num_nodes=n_nodes_ring,
    topology='ring'
)

print(f"Ring Topology (C_{n_nodes_ring}) Network Properties:")
print(f"{'─' * 50}")
print(f"Nodes (|V|):                    {ring_graph.number_of_nodes()}")
print(f"Edges (|E|):                    {ring_graph.number_of_edges()}")
print(f"Expected edges for cycle:       {n_nodes_ring}")
print(f"Density (ρ):                    {nx.density(ring_graph):.4f}")
print(f"Average degree:                 {2 * ring_graph.number_of_edges() / ring_graph.number_of_nodes():.2f}")
print(f"Diameter:                       {nx.diameter(ring_graph)}")
print(f"Average clustering coefficient: {nx.average_clustering(ring_graph):.4f}")
print(f"Is connected:                   {nx.is_connected(ring_graph)}")

# Compute degree distribution
degrees_ring = [ring_graph.degree(n) for n in ring_graph.nodes()]
print(f"\nDegree distribution: min={min(degrees_ring)}, max={max(degrees_ring)}, mean={np.mean(degrees_ring):.2f}")

# Verify it's a valid cycle
print(f"\nIs Eulerian (all nodes even degree): {all(d % 2 == 0 for d in degrees_ring)}")

### 3.3 Tree Topology Implementation

We construct a hierarchical tree with balanced structure using breadth-first generation.

**Algorithm**:
```
Input: n (number of nodes), branching_factor (b)
1. Initialize root node r
2. For each level k:
   - For each parent node p at level k:
     - Create b children at level k+1
3. Stop when |V| = n
4. Assign electrical parameters
Output: Balanced tree with height ≈ log_b(n)
```

**Complexity**:
- Balanced tree height: $h = \lceil \log_b n \rceil$
- Maximum diameter: $2h = O(\log n)$
- Minimum diameter: $h$ (deep narrow trees)

In [ ]:
# 3.3 TREE TOPOLOGY CONSTRUCTION
# Create a hierarchical balanced tree

n_nodes_tree = 16
tree_graph, tree_metadata = create_microgrid(
    num_nodes=n_nodes_tree,
    topology='tree'
)

print(f"Tree Topology Network Properties:")
print(f"{'─' * 50}")
print(f"Nodes (|V|):                    {tree_graph.number_of_nodes()}")
print(f"Edges (|E|):                    {tree_graph.number_of_edges()}")
print(f"Expected edges for tree:        {n_nodes_tree - 1}")
print(f"Density (ρ):                    {nx.density(tree_graph):.4f}")
print(f"Average degree:                 {2 * tree_graph.number_of_edges() / tree_graph.number_of_nodes():.2f}")
print(f"Diameter:                       {nx.diameter(tree_graph)}")
print(f"Average clustering coefficient: {nx.average_clustering(tree_graph):.4f}")
print(f"Is connected:                   {nx.is_connected(tree_graph)}")
print(f"Is acyclic (tree):              {nx.is_tree(tree_graph)}")

# Compute degree distribution
degrees_tree = [tree_graph.degree(n) for n in tree_graph.nodes()]
print(f"\nDegree distribution: min={min(degrees_tree)}, max={max(degrees_tree)}, mean={np.mean(degrees_tree):.2f}")

# Find leaves and root
leaves = [n for n in tree_graph.nodes() if tree_graph.degree(n) == 1]
print(f"Number of leaf nodes: {len(leaves)} ({100*len(leaves)/tree_graph.number_of_nodes():.1f}%)")

## 4. Comparative Topology Analysis

### 4.1 Network Metrics Comparison

We compare the three topologies across key electrical and structural metrics.

In [ ]:
# 4.1 COMPREHENSIVE TOPOLOGY COMPARISON

topologies = {
    'Mesh': (mesh_graph, mesh_metadata),
    'Ring': (ring_graph, ring_metadata),
    'Tree': (tree_graph, tree_metadata)
}

metrics_data = []

for topo_name, (graph, metadata) in topologies.items():
    n = graph.number_of_nodes()
    m = graph.number_of_edges()
    
    # Structural metrics
    density = nx.density(graph)
    diameter = nx.diameter(graph) if nx.is_connected(graph) else float('inf')
    clustering = nx.average_clustering(graph)
    avg_degree = 2 * m / n
    
    # Redundancy metric: average number of paths
    # For simplicity, use edge connectivity
    if nx.is_connected(graph):
        edge_conn = nx.edge_connectivity(graph)
    else:
        edge_conn = 0
    
    # Path length (average shortest path)
    if nx.is_connected(graph):
        avg_path = nx.average_shortest_path_length(graph)
    else:
        avg_path = float('inf')
    
    metrics_data.append({
        'Topology': topo_name,
        'Nodes (|V|)': n,
        'Edges (|E|)': m,
        'Density (ρ)': f"{density:.4f}",
        'Avg Degree': f"{avg_degree:.2f}",
        'Diameter': diameter,
        'Avg Path Length': f"{avg_path:.2f}",
        'Clustering Coeff.': f"{clustering:.4f}",
        'Edge Connectivity': edge_conn,
        'Cost (|E| relative)': f"{m}"
    })

metrics_df = pd.DataFrame(metrics_data)
print("\n" + "=" * 120)
print("TOPOLOGY COMPARATIVE ANALYSIS")
print("=" * 120)
print(metrics_df.to_string(index=False))
print("=" * 120)

### 4.2 Electrical Impedance & Admittance Matrix

We construct and analyze the nodal admittance matrices for each topology.

**Theory**: For a network with line impedances $Z_{ij}$, the admittance is $Y_{ij} = 1/Z_{ij}$. The nodal admittance matrix satisfies:

$$\mathbf{I} = \mathbf{Y}_{bus} \mathbf{V}$$

where $\mathbf{I}$ is the vector of nodal current injections and $\mathbf{V}$ is the vector of nodal voltages.

In [ ]:
# 4.2 NODAL ADMITTANCE MATRIX CONSTRUCTION

def construct_admittance_matrix(graph: nx.Graph) -> np.ndarray:
    """
    Construct the nodal admittance matrix (Y_bus) for the network.
    
    Parameters:
    -----------
    graph : nx.Graph
        Network graph with impedance attributes
    
    Returns:
    --------
    Y_bus : np.ndarray (complex)
        Nodal admittance matrix
    """
    n = graph.number_of_nodes()
    nodes = sorted(list(graph.nodes()))
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    # Initialize Y_bus
    Y_bus = np.zeros((n, n), dtype=complex)
    
    # Fill Y_bus elements
    for i, j in graph.edges():
        idx_i = node_to_idx[i]
        idx_j = node_to_idx[j]
        
        # Get impedance from edge attributes
        R = float(graph.edges[i, j].get('resistance', 0.05))
        X = float(graph.edges[i, j].get('reactance', 0.10))
        Z = complex(R, X)
        Y = 1.0 / Z if Z != 0 else 0
        
        # Off-diagonal elements: Y_ij = -Y (mutual admittance)
        Y_bus[idx_i, idx_j] -= Y
        Y_bus[idx_j, idx_i] -= Y
        
        # Diagonal elements: sum of connected admittances
        Y_bus[idx_i, idx_i] += Y
        Y_bus[idx_j, idx_j] += Y
    
    return Y_bus

# Compute Y_bus for each topology
Y_mesh = construct_admittance_matrix(mesh_graph)
Y_ring = construct_admittance_matrix(ring_graph)
Y_tree = construct_admittance_matrix(tree_graph)

print("Nodal Admittance Matrix (Y_bus) Analysis")
print("=" * 70)
print(f"\nMESH Topology (K_{mesh_graph.number_of_nodes()}):")
print(f"  Matrix shape: {Y_mesh.shape}")
print(f"  Rank: {np.linalg.matrix_rank(Y_mesh)}")
print(f"  Condition number: {np.linalg.cond(Y_mesh):.2e}")
print(f"  Sparsity: {np.count_nonzero(Y_mesh) / Y_mesh.size * 100:.1f}%")
print(f"  Frobenius norm ||Y_bus||_F: {np.linalg.norm(Y_mesh, 'fro'):.4f}")

print(f"\nRING Topology (C_{ring_graph.number_of_nodes()}):")
print(f"  Matrix shape: {Y_ring.shape}")
print(f"  Rank: {np.linalg.matrix_rank(Y_ring)}")
print(f"  Condition number: {np.linalg.cond(Y_ring):.2e}")
print(f"  Sparsity: {np.count_nonzero(Y_ring) / Y_ring.size * 100:.1f}%")
print(f"  Frobenius norm ||Y_bus||_F: {np.linalg.norm(Y_ring, 'fro'):.4f}")

print(f"\nTREE Topology:")
print(f"  Matrix shape: {Y_tree.shape}")
print(f"  Rank: {np.linalg.matrix_rank(Y_tree)}")
print(f"  Condition number: {np.linalg.cond(Y_tree):.2e}")
print(f"  Sparsity: {np.count_nonzero(Y_tree) / Y_tree.size * 100:.1f}%")
print(f"  Frobenius norm ||Y_bus||_F: {np.linalg.norm(Y_tree, 'fro'):.4f}")

print("\n" + "=" * 70)
print("Interpretation:")
print("  - Condition number: Lower is better (numerically stable)")
print("  - Sparsity: Mesh=dense, Tree=sparse (affects solver efficiency)")
print("  - Rank: Should equal n for connected networks")

## 5. Visualization of Network Topologies

We create publication-quality visualizations of each topology with node attributes.

In [ ]:
# 5. TOPOLOGY VISUALIZATION

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

topology_list = [
    (mesh_graph, 'Mesh (K_8)', axes[0], 'spring'),
    (ring_graph, 'Ring (C_12)', axes[1], 'circular'),
    (tree_graph, 'Tree', axes[2], 'spring')
]

for graph, title, ax, layout_type in topology_list:
    # Choose layout
    if layout_type == 'circular':
        pos = nx.circular_layout(graph)
    else:
        pos = nx.spring_layout(graph, k=0.5, iterations=50, seed=42)
    
    # Node coloring based on degree (higher degree = more connected)
    degrees = [graph.degree(n) for n in graph.nodes()]
    
    # Draw edges
    nx.draw_networkx_edges(
        graph, pos, ax=ax, alpha=0.3, width=1.5, edge_color='gray'
    )
    
    # Draw nodes with color gradient
    nodes = nx.draw_networkx_nodes(
        graph, pos, ax=ax,
        node_color=degrees,
        node_size=300,
        cmap='YlOrRd',
        edgecolors='black',
        linewidths=1.5
    )
    
    # Draw labels
    nx.draw_networkx_labels(
        graph, pos, ax=ax, font_size=9, font_weight='bold'
    )
    
    # Formatting
    ax.set_title(f'{title}\n({graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges)', 
                 fontsize=12, fontweight='bold')
    ax.axis('off')
    
    # Add colorbar for degree
    cbar = plt.colorbar(nodes, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Node Degree', fontsize=9)

plt.suptitle('DC Microgrid Topologies: Structural Comparison', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('topology_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Topology visualization saved")

## 6. Power Flow Analysis & Network Response

### 6.1 Power Flow Problem Formulation

Given a network with specified power injections at each bus, we solve for voltages:

**Nodal Power Balance Equations**:

$$P_i + jQ_i = V_i \sum_{k=1}^{n} |Y_{ik}| |V_k| e^{j(\theta_{ik} - \delta_i + \delta_k)}$$

where:
- $P_i, Q_i$: Real and reactive power injection at node $i$
- $V_i, \delta_i$: Voltage magnitude and angle at node $i$
- $|Y_{ik}|, \theta_{ik}$: Magnitude and angle of admittance element

**For DC Approximation** (ignores reactive power):

$$P_i = V_i \sum_{k=1}^{n} Y_{ik} V_k \cos(\delta_i - \delta_k)$$

This is an iterative nonlinear problem solved via Newton-Raphson method.

In [ ]:
# 6.1 POWER FLOW ANALYSIS FOR EACH TOPOLOGY

print("Power Flow Analysis & Network Response")
print("=" * 80)

for graph, topo_name in [(mesh_graph, 'Mesh'), (ring_graph, 'Ring'), (tree_graph, 'Tree')]:
    # Assign power injections
    nodes = list(graph.nodes())
    n = len(nodes)
    
    # Set generation at nodes 0-2, loads at remaining nodes
    for i, node in enumerate(nodes):
        if i < 3:
            graph.nodes[node]['generation'] = 20.0 + np.random.uniform(-5, 5)
            graph.nodes[node]['load'] = 5.0
        else:
            graph.nodes[node]['generation'] = 0.0
            graph.nodes[node]['load'] = 15.0 + np.random.uniform(-3, 3)
    
    # Run power flow
    try:
        pf_results = run_powerflow_analysis(graph)
        constraints = check_constraints(graph, pf_results)
        
        print(f"\n{topo_name.upper()} TOPOLOGY Results:")
        print(f"{'-' * 80}")
        print(f"Convergence Status:           {'CONVERGED' if pf_results.get('converged') else 'DIVERGED'}")
        print(f"Iterations:                   {pf_results.get('iterations', 'N/A')}")
        print(f"Total Network Loss:           {pf_results.get('total_loss', 0):.4f} MW")
        print(f"Voltage violations:           {len(constraints.get('voltage_violations', []))}")
        print(f"Thermal violations:           {len(constraints.get('thermal_violations', []))}")
        
        # Voltage statistics
        voltages = pf_results.get('voltages', np.ones(n))
        if isinstance(voltages, np.ndarray):
            print(f"\nVoltage Statistics (pu):")
            print(f"  Min voltage:                {np.min(voltages):.4f}")
            print(f"  Max voltage:                {np.max(voltages):.4f}")
            print(f"  Mean voltage:               {np.mean(voltages):.4f}")
            print(f"  Std deviation:              {np.std(voltages):.4f}")
    
    except Exception as e:
        print(f"\n{topo_name.upper()} TOPOLOGY: Power flow error: {e}")

print("\n" + "=" * 80)

## 7. Key Insights & Design Implications

### 7.1 Topology Selection Criteria

| Criterion | Mesh | Ring | Tree |
|-----------|------|------|------|
| **Redundancy** | Excellent | Good | None |
| **Voltage Regulation** | Superior | Moderate | Poor |
| **Cost** | Very High ($O(n^2)$) | Low ($O(n)$) | Very Low ($O(n)$) |
| **Loss Profile** | Distributed | Concentrated | Severe at root |
| **Scalability** | Poor ($n < 20$) | Moderate ($n < 100$) | Excellent ($n > 1000$) |
| **Best for** | Critical systems | Medium networks | Distribution networks |

### 7.2 Design Trade-offs

1. **Reliability vs. Cost**: Mesh offers excellent redundancy but is prohibitively expensive
2. **Voltage Quality vs. Scalability**: Tree topologies scale well but suffer from voltage drop
3. **Fault Tolerance vs. Simplicity**: Ring/Tree have critical failure points

### 7.3 Practical Applications

- **Mesh**: Backup/emergency power, high-reliability data centers
- **Ring**: Industrial parks, city microgrid backbones
- **Tree**: Distribution feeders, residential/commercial networks

### 7.4 EV Charging Integration

For EV-centric microgrids:
- **Charging stations**: Typically distributed (mesh/ring advantages)
- **Voltage regulation**: CC-CV charging requires stable voltages (favor ring/mesh)
- **Cost constraints**: Tree topology with voltage regulators (compromise solution)
- **Fault current**: Multiple sources in mesh increase fault levels (protection coordination complexity)

In [ ]:
# 7. SUMMARY STATISTICS

print("\n" + "=" * 80)
print("NETWORK DESIGN SUMMARY")
print("=" * 80)
print(f"""
This notebook demonstrated the design and analysis of three canonical DC microgrid
topologies for EV-centric applications:

1. MESH (Complete Graph K_n):
   - Maximum connectivity (n-1 edges per node)
   - Excellent fault tolerance and voltage regulation
   - Cost: O(n²) - prohibitive for large networks
   - Best for: Small critical systems requiring high reliability

2. RING (Cycle Graph C_n):
   - Moderate connectivity (2 edges per node)
   - Natural circular architecture for feeders
   - Cost: O(n) - efficient
   - Best for: Medium-sized networks with balanced load distribution

3. TREE (Minimum Spanning Tree):
   - Minimal connectivity (n-1 edges total)
   - Cost: O(n) - optimal for scalability
   - Hierarchical structure (source → distribution)
   - Best for: Large distribution networks with clear supply points

Key Metrics:
  - Density: Mesh >> Ring >> Tree
  - Robustness: Mesh > Ring > Tree  
  - Scalability: Tree > Ring > Mesh
  - Voltage regulation: Mesh > Ring > Tree

For EV charging integration, a hybrid approach is recommended:
  - Ring backbone with tree distribution feeders
  - Voltage regulators at feeder heads
  - Multi-source capability via CC-CV charging stations
""")
print("=" * 80)